In [75]:
import pandas as pd

In [76]:
# usercols charge les 05 colonnes utile & parse_dates convertit les 03 colonnes de dates en vraie date 
orders = pd.read_csv(
    '../data/raw/olist_orders_dataset.csv',
    usecols=['order_id', 'order_status', 'order_purchase_timestamp',
             'order_delivered_customer_date', 'order_estimated_delivery_date'],
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date']
)

In [77]:
# lit les numèros de commande avec les prix
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv',
                    usecols=['order_id', 'price'])

In [78]:
# Lit les avis, avec la note et la date de reponse celà permettra de garder l'avis le plus recent
reviews = pd.read_csv(
    '../data/raw/olist_order_reviews_dataset.csv',
    usecols=['review_id', 'order_id', 'review_score', 'review_answer_timestamp'],
    parse_dates=['review_answer_timestamp']
)

In [79]:
print(orders.shape, items.shape, reviews.shape)

(99441, 5) (112650, 2) (99224, 4)


calcule du premier kpi sur le chiffre d'affaire

In [80]:
# garde en un seul endroit la liste des statuts qui ne compte pas dans le ca
STATUTS_EXCLUS = ['canceled', 'unavailable']

In [81]:
#ajoute le statut à chaque article, en reliant les deux tableaux par order_id et how='left' garde tous les articles
lignes = items.merge(orders[['order_id', 'order_status']], on='order_id', how='left')
print(len(lignes))

112650


In [82]:
#garde les articles dont le statut n’est pas dans la liste des exclus & isin garde l'appartenance et ~ pour non
valides = lignes[~lignes['order_status'].isin(STATUTS_EXCLUS)]
ca_total = round(valides['price'].sum(), 2) # additionne les prix et garde la valeur à deux chiffres apès la virgule
print(len(valides), ca_total)

112101 13494400.74


2ème kpi sur les commendes des articles livrés

In [83]:
# garde les commandes donc le statut est delivered 
livrees = orders[orders['order_status'] == 'delivered']
commandes_livrees = len(livrees)

In [84]:
# retire de ce tableau les commandes sans date de livraison
livrees_avec_date = livrees.dropna(subset=['order_delivered_customer_date'])
print(commandes_livrees, len(livrees_avec_date))

96478 96470


calcul du kpi numero 3 le delai moyen de livraison 

In [85]:
# ici, on calcule pour chaque commande livrée avec date, la durée entre l’achat et la livraison au client
duree = livrees_avec_date['order_delivered_customer_date'] - livrees_avec_date['order_purchase_timestamp']

In [86]:
# convertit la durée en secondespuis la divise par 86400 pourqu'on puisse avoir les dates en decimales
jours_decimaux = duree.dt.total_seconds() / 86400
jours_entiers = duree.dt.days

In [87]:
print(round(jours_decimaux.mean(), 3), round(jours_entiers.mean(), 3))

12.558 12.094


test d'hypothèse

In [88]:
# calcule, pour chaque commande, le délai en jours entiers, vide si pas de date de livraison
delai_commande = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
delais = orders[['order_id']].assign(delai=delai_commande) # le range dans un petit tableau avec le numéro de commande

In [89]:
articles = items[['order_id']].merge(delais, on='order_id', how='left')
print(len(articles), articles['delai'].notna().sum(), round(articles['delai'].mean(), 3))

112650 110196 12.008


kpi 04 taux de retard

In [90]:
# Calcule, pour chaque commande, le retard en jours entiers, négatif = en avance, vide si pas livrée
retard_commande = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
retards = orders[['order_id']].assign(retard=retard_commande) 



In [91]:
art = items[['order_id']].merge(retards, on='order_id', how='left').dropna(subset=['retard']) # donne à chaque article le retard de sa commande et retire ceux sans date de livraison
taux_retard = round(float((art['retard'] > 0).mean()) * 100, 2) # calcule la part des articles dont le retard est supérieur à 0, .mean() d’une colonne vrai/faux donne la proportion
print(len(art), taux_retard) # affiche le nombre d’articles attendu et le taux

110196 6.59


kpi 05 la note moyenne

In [92]:
avis = (reviews.sort_values(['order_id', 'review_answer_timestamp', 'review_id']) # trie les avis par commande, puis du plus ancien au plus récen
               .drop_duplicates('order_id', keep='last')) # garde un seul avis par commande, le dernier (keep='last')
assert len(avis) == 98673 # vérifie qu’il reste 98 673 avis

In [93]:
notes_articles = items[['order_id']].merge(avis[['order_id', 'review_score']], on='order_id', how='left') # donne à chaque article la note de sa commande, vide s’il n’y a pas d’avis
print(len(notes_articles), notes_articles['review_score'].notna().sum()) # affiche le nombre d’articles attendu et ceux qui ont une note
print(round(avis['review_score'].mean(), 3), round(notes_articles['review_score'].mean(), 3)) # affiche la moyenne par commande, puis par article

112650 111708
4.086 4.033


In [94]:
import json
from datetime import date

In [95]:
# rassemble les resultats dans un dixtionnaire, une clé par kpi plus la date de calcul
kpis = {
    "date_calcul": date.today().isoformat(),
    "ca_total_hors_annule": float(ca_total),
    "commandes_livrees": int(commandes_livrees),
    "delai_moyen_livraison": round(float(articles['delai'].mean()), 3),
    "taux_retard": float(taux_retard),
    "note_moyenne": round(float(notes_articles['review_score'].mean()), 3),
    "note_moyenne_par_commande": round(float(avis['review_score'].mean()), 3),
}

In [97]:
# écrivent le dictionnaire dans le fichier, avec une mise en forme lisible
with open('../docs/kpis_reference.json', 'w', encoding='utf-8') as f:
    json.dump(kpis, f, indent=2, ensure_ascii=False)
print(open('../docs/kpis_reference.json', encoding='utf-8').read()) # relit le fichier et l’affiche, pour vérifier

{
  "date_calcul": "2026-10-20",
  "ca_total_hors_annule": 13494400.74,
  "commandes_livrees": 96478,
  "delai_moyen_livraison": 12.008,
  "taux_retard": 6.59,
  "note_moyenne": 4.033,
  "note_moyenne_par_commande": 4.086
}
